# VulBERTa LoRA Notebook

A standalone notebook for loading a VulBERTa checkpoint and applying PEFT LoRA adapters.

Default checkpoint: `claudios/VulBERTa-MLP-D2A`.
Set `VULBERTA_MODEL` to use a local path or a different Hugging Face repo.

In [11]:
# Remove incompatible torchao before loading PEFT.
# This keeps the full workflow inside the notebook.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torchao') is not None:
    print('Removing incompatible torchao package...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

# Uncomment if the notebook kernel does not already have these packages.
# %pip install -q --upgrade transformers accelerate peft datasets sentencepiece

import os
import torch
from dataclasses import dataclass

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

torch: 2.10.0+cpu
cuda available: False


## Load VulBERTa

This notebook defaults to a VulBERTa checkpoint and keeps the LoRA targets 

In [12]:
DEFAULT_MODEL_NAME = os.environ.get('VULBERTA_MODEL', 'claudios/VulBERTa-MLP-D2A')
DEFAULT_MAX_LENGTH = 128
DEFAULT_NUM_LABELS = 2

@dataclass(frozen=True)
class LoRASettings:
    r: int = 8
    alpha: int = 32
    dropout: float = 0.1
    num_labels: int = DEFAULT_NUM_LABELS
    max_length: int = DEFAULT_MAX_LENGTH


def suggest_target_modules(model):
    module_names = {name.split('.')[-1] for name, _ in model.named_modules()}
    preferred_groups = [
        ['query', 'key', 'value'],
        ['query', 'value'],
        ['q_proj', 'k_proj', 'v_proj'],
        ['q_proj', 'v_proj'],
    ]
    for group in preferred_groups:
        if all(module_name in module_names for module_name in group):
            return group
    fallback = [name for name in ['query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj'] if name in module_names]
    return fallback or ['query', 'value']


def load_vulberta(model_name=DEFAULT_MODEL_NAME, num_labels=DEFAULT_NUM_LABELS):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    model.config.label2id = {str(index): index for index in range(num_labels)}
    model.config.id2label = {index: str(index) for index in range(num_labels)}
    return tokenizer, model

## Apply LoRA

The adapter targets the attention projection names exposed by the VulBERTa backbone.

In [13]:
def build_lora_model(base_model, settings=LoRASettings(), target_modules=None):
    modules = list(target_modules) if target_modules is not None else suggest_target_modules(base_model)
    config = LoraConfig(
        r=settings.r,
        lora_alpha=settings.alpha,
        lora_dropout=settings.dropout,
        target_modules=modules,
        bias='none',
        task_type=TaskType.SEQ_CLS,
    )
    return get_peft_model(base_model, config)


def save_adapter(model, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)


def load_adapter_for_inference(model_name, adapter_dir, num_labels=DEFAULT_NUM_LABELS):
    tokenizer, base_model = load_vulberta(model_name=model_name, num_labels=num_labels)
    adapted_model = PeftModel.from_pretrained(base_model, adapter_dir)
    adapted_model.eval()
    return tokenizer, adapted_model

## Smoke Test

Load the base checkpoint, wrap it with LoRA, and run a tiny prediction example.

In [14]:
def predict(tokenizer, model, texts, max_length=DEFAULT_MAX_LENGTH):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    encoded = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        return_tensors='pt',
        max_length=max_length,
    )
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)
    return predictions.cpu(), probabilities.cpu()


def smoke_test(model_name=DEFAULT_MODEL_NAME):
    tokenizer, base_model = load_vulberta(model_name=model_name)
    lora_model = build_lora_model(base_model)

    trainable_params = sum(parameter.numel() for parameter in lora_model.parameters() if parameter.requires_grad)
    total_params = sum(parameter.numel() for parameter in lora_model.parameters())
    print(f'Loaded: {model_name}')
    print(f'Target modules: {suggest_target_modules(base_model)}')
    print(f'Trainable params: {trainable_params} / {total_params} ({100 * trainable_params / total_params:.2f}%)')

    sample_texts = [
        'Potential buffer overflow in copy routine.',
        'Helper function for formatting dates.',
    ]
    predictions, probabilities = predict(tokenizer, lora_model, sample_texts)
    print('Predictions:', predictions.tolist())
    print('Probabilities:', probabilities.tolist())


smoke_test()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded: claudios/VulBERTa-MLP-D2A
Target modules: ['query', 'key', 'value']
Trainable params: 1034498 / 125871364 (0.82%)
Predictions: [1, 0]
Probabilities: [[0.20570647716522217, 0.7942935228347778], [0.7996673583984375, 0.2003326117992401]]
